# f3_m00_ejecucion.ipynb

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | Fase 3 — Features |
| **Módulo** | M00 — Ejecución |

---

## 🎯 Qué hace

Orquestador de Fase 3 — Features. Lanza cada notebook en secuencia usando
`nbconvert --execute` y actualiza una tabla de progreso con `ipywidgets` en tiempo real.
Si un notebook falla, lo marca con ❌ y continúa con el siguiente.
Genera un HTML resumen con el estado de ejecución.

## 📋 Requisitos

- Todos los notebooks de Fase 3 — Features en `notebooks/fase3_features/`
- `src/utils/orquestador.py` con `orquestador_fase()` y `ejecutar_notebook()`
- Entorno: `tfm_abandono` (nbconvert, ipywidgets)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase3/m00_ejecucion.html` | Resumen HTML del estado de ejecución |

## 🔄 Flujo

```
Para cada notebook en NOTEBOOKS:
    ↓ nbconvert --execute (timeout=3600s)
    ↓ Actualizar tabla de progreso ipywidgets
    ↓ Registrar errores si los hay
→ docs/html/fase3/m00_ejecucion.html
```

## ➡️ Siguiente

No aplica — este notebook es el último en ejecutarse.
Revisar los HTMLs generados en `docs/html/fase3/`.


In [1]:
# 1. CONFIGURACIÓN DE RUTAS
import sys
from pathlib import Path

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_NOTEBOOKS = Path.cwd()

print(f'ROOT: {ROOT}')
print(f'DIR_NOTEBOOKS: {DIR_NOTEBOOKS}')


ROOT: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_NOTEBOOKS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks\fase3_features


In [2]:
# 2. IMPORTS
from src.utils.orquestador import orquestador_fase, ejecutar_notebook
from src.html.render import render_pagina_desde_fichero


In [3]:
# 3. LISTA DE NOTEBOOKS A EJECUTAR
NOTEBOOKS = [
    ('m00', 'f3_m00_indice.ipynb'),
    ('m01', 'f3_m01_validacion.ipynb'),
    ('m02', 'f3_m02_agregacion.ipynb'),
    ('m03', 'f3_m03_features.ipynb'),
    ('m04', 'f3_m04_index.ipynb'),
    ('m04a', 'f3_m04a_automl_target.ipynb'),
    ('m04b', 'f3_m04b_eda_target.ipynb'),
    ('m05', 'f3_m05_target_export.ipynb'),
    ('m06', 'f3_m06_alerta_temprana.ipynb'),
    ('m07', 'f3_m07_validacion.ipynb'),
    ('m08', 'f3_m08_auditoria.ipynb'),
]

TIMEOUT_POR_CELDA = 3600  # 1 hora por celda

print(f'{len(NOTEBOOKS)} notebooks configurados.')


11 notebooks configurados.


In [4]:
# 4. INSTANCIAR ORQUESTADOR Y MOSTRAR TABLA
orch = orquestador_fase("Fase 3 — Features")
orch.mostrar()


HTML(value='\n        <div style="font-family:monospace; margin:12px 0">\n            <div style="font-size:15…

In [5]:
# 5. EJECUTAR NOTEBOOKS EN SECUENCIA
errores = []

for modulo_id, nombre_nb in NOTEBOOKS:
    ruta_nb = DIR_NOTEBOOKS / nombre_nb

    if not ruta_nb.exists():
        print(f'⚠️  {nombre_nb} no encontrado — saltando.')
        orch.error(modulo_id)
        errores.append((modulo_id, nombre_nb, 'fichero no encontrado'))
        continue

    orch.iniciar(modulo_id)
    ok = ejecutar_notebook(ruta_nb, timeout=TIMEOUT_POR_CELDA)

    if ok:
        orch.ok(modulo_id)
    else:
        orch.error(modulo_id)
        errores.append((modulo_id, nombre_nb, 'error en ejecución'))

n_ok = len(NOTEBOOKS) - len(errores)
print(f'\n✅ Completados: {n_ok}/{len(NOTEBOOKS)}')
if errores:
    print('\n❌ Módulos con error:')
    for mid, nb, motivo in errores:
        print(f'   {mid}: {nb} — {motivo}')



✅ Completados: 11/11


In [6]:
# 6. GENERAR HTML DE ESTE MÓDULO
n_ok    = len(NOTEBOOKS) - len(errores)
n_total = len(NOTEBOOKS)
pct     = int(n_ok / n_total * 100) if n_total else 0

filas_resumen = ''
for mid, nb in NOTEBOOKS:
    fue_error = any(e[0] == mid for e in errores)
    icono = '❌' if fue_error else '✅'
    bg    = '#fff5f5' if fue_error else '#f0fff4'
    filas_resumen += f'''
    <tr style="background:{bg}">
        <td style="padding:8px 12px">{icono}</td>
        <td style="padding:8px 12px; font-family:monospace; font-size:12px">{mid}</td>
        <td style="padding:8px 12px; font-size:13px">{nb}</td>
    </tr>'''

barra_color = '#38a169' if pct == 100 else '#3182ce'

contenido_html = f'''
<h2 style="color:#2d3748">▶️ Orquestador de Fase 3 — Features</h2>
<p style="color:#4a5568; font-size:14px">
    Ejecuta todos los notebooks de Fase 3 — Features en secuencia usando
    <code>nbconvert --execute</code> con barra de progreso visual.
</p>
<div style="margin:20px 0; padding:16px; background:#f7fafc; border-radius:8px">
    <div style="font-size:14px; color:#4a5568; margin-bottom:8px">
        Progreso: {n_ok}/{n_total} notebooks ({pct}%)
    </div>
    <div style="background:#e2e8f0; border-radius:4px; height:10px; width:100%">
        <div style="background:{barra_color}; border-radius:4px; height:10px; width:{pct}%"></div>
    </div>
</div>
<table style="width:100%; border-collapse:collapse; font-size:13px">
    <thead>
        <tr style="background:#edf2f7">
            <th style="padding:8px 12px; text-align:left">Estado</th>
            <th style="padding:8px 12px; text-align:left">ID</th>
            <th style="padding:8px 12px; text-align:left">Notebook</th>
        </tr>
    </thead>
    <tbody>{filas_resumen}</tbody>
</table>
'''

html_completo = render_pagina_desde_fichero('f3_m00_ejecucion.ipynb', contenido_html)
ruta_html = ROOT / 'docs' / 'html' / 'fase3' / 'm00_ejecucion.html'
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')


✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m00_ejecucion.html
